# PipelineWatch-NG — Module 2: Processing and Feature Engineering
### Trans Niger Pipeline corridor · Niger Delta, Nigeria

**Run Module 1 first**, then open this notebook.

| Step | What it does |
|------|--------------|
| Sentinel-2 NDVI/NDWI | Vegetation dieback and water contamination along pipeline ROW |
| SAR change detection | Recent minus baseline — isolates new spill activity since 2023 |
| DBSCAN clustering | Groups 50 fire hotspots into distinct refinery site candidates |
| Feature table | Labelled CSV ready for ML training in Module 3 |

---

## Cell 1 — Imports and GEE connection

In [1]:
import ee
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.cluster import DBSCAN

GEE_PROJECT_ID = "pipelinewatch-ng"

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"GEE connected: {GEE_PROJECT_ID}")
    print(f"Test: {ee.Number(42).getInfo()}")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print("Authenticated.")


C:\Users\taylo\anaconda3\envs\pipelinewatch\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


GEE connected: pipelinewatch-ng
Test: 42


## Cell 2 — Config

In [2]:
ROI_BOUNDS     = [6.50, 5.00, 7.20, 5.80]
ROI_CENTER     = [5.40, 6.85]
ROI_ZOOM       = 9
BASELINE_START = "2023-01-01"
BASELINE_END   = "2023-06-30"
RECENT_START   = "2024-01-01"
RECENT_END     = "2024-06-30"
SAR_SIGMA      = 1.5
FIRMS_K        = 330.0
CACHE_DIR      = "../data/cached"
OUTPUT_DIR     = "../outputs"
os.makedirs(CACHE_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
ROI = ee.Geometry.Rectangle(ROI_BOUNDS)
print("Config ready.")


Config ready.


## Cell 3 — Load Module 1 cached outputs

In [3]:
def load_geojson(filename):
    path = os.path.join(CACHE_DIR, filename)
    if not os.path.exists(path):
        print('  WARNING: ' + filename + ' not found — run Module 1 first')
        return None, 0
    with open(path) as f:
        gj = json.load(f)
    n = len(gj.get('features', []))
    print('  ' + filename + ': ' + str(n) + ' features')
    return gj, n

print('Loading Module 1 outputs...')
sar_gj,   n_sar   = load_geojson('sar_dark_spots.geojson')
firms_gj, n_firms = load_geojson('fire_hotspots.geojson')
so2_gj,   n_so2   = load_geojson('so2_anomalies.geojson')

# Convert FIRMS to DataFrame for DBSCAN
rows = []
if firms_gj and n_firms > 0:
    for feat in firms_gj['features']:
        coords = feat['geometry']['coordinates']
        props  = feat['properties']
        rows.append({
            'lon':           coords[0],
            'lat':           coords[1],
            'T21_max_K':     props.get('T21_max_K', 0) or 0,
            'fire_count':    props.get('fire_count', 0) or 0,
            'likely_source': props.get('likely_source', 'unknown'),
            'confidence':    props.get('confidence', 'low'),
        })

df_firms = pd.DataFrame(rows)
print('FIRMS DataFrame: ' + str(len(df_firms)) + ' hotspots')
if len(df_firms) > 0:
    print(df_firms[['lon','lat','T21_max_K','fire_count','likely_source']].head())

Loading Module 1 outputs...
  sar_dark_spots.geojson: 3 features
  fire_hotspots.geojson: 50 features
  so2_anomalies.geojson: 0 features
FIRMS DataFrame: 50 hotspots
        lon       lat   T21_max_K  fire_count          likely_source
0  6.504926  5.086710  349.299988    2.850496  agricultural_or_other
1  6.508775  5.711601  336.000000    3.562633  agricultural_or_other
2  6.508294  5.795818  338.200012    3.000000  agricultural_or_other
3  6.520085  5.688020  335.399994    3.000000  agricultural_or_other
4  6.518342  5.700798  336.000000    5.846860  agricultural_or_other


## Cell 4 — Sentinel-2 and SAR functions
Function definitions — nothing runs yet.

**NDVI** = (NIR - Red) / (NIR + Red) — healthy mangrove: 0.4-0.8, oil-damaged: <0.2
**NDWI** = (Green - NIR) / (Green + NIR) — open water: positive, vegetation: negative
**SAR change** = recent VV minus baseline VV — new darkening < -3dB = new spill candidate

In [4]:
# ── Sentinel-2 functions ────────────────────────────────────────────

def get_s2_collection(roi, start, end, cloud_pct=30):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct))
        .select(["B2","B3","B4","B8","B11","QA60"])
    )

def mask_s2_clouds(image):
    qa          = image.select("QA60")
    cloud_mask  = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus_mask = qa.bitwiseAnd(1 << 11).eq(0)
    return (
        image.updateMask(cloud_mask.And(cirrus_mask))
        .divide(10000)
        .copyProperties(image, ["system:time_start"])
    )

def compute_s2_indices(image):
    nir   = image.select("B8")
    red   = image.select("B4")
    green = image.select("B3")
    swir  = image.select("B11")
    ndvi  = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    ndwi  = green.subtract(nir).divide(green.add(nir)).rename("NDWI")
    ndmi  = nir.subtract(swir).divide(nir.add(swir)).rename("NDMI")
    return image.addBands(ndvi).addBands(ndwi).addBands(ndmi)

def build_s2_composite(roi, start, end, cloud_pct=30):
    col   = get_s2_collection(roi, start, end, cloud_pct)
    count = col.size().getInfo()
    print(f"  S2 scenes ({start} to {end}): {count}")
    if count == 0:
        print("  WARNING: no S2 scenes — try increasing cloud_pct")
        return None, 0
    clean = col.map(mask_s2_clouds).map(compute_s2_indices)
    return clean.median().clip(roi), count

# ── SAR functions ───────────────────────────────────────────────────

def get_s1_collection(roi, start, end):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi)
        .filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
        .select(["VV", "VH"])
    )

def compute_sar_features(image, roi, sigma=1.5):
    vv    = image.select("VV")
    vh    = image.select("VH")
    stats = vv.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=roi, scale=100, maxPixels=1e9, bestEffort=True
    )
    mean_vv   = ee.Number(stats.get("VV_mean"))
    std_vv    = ee.Number(stats.get("VV_stdDev"))
    threshold = mean_vv.subtract(std_vv.multiply(sigma))
    dark_mask = vv.lt(threshold).rename("dark_spot_mask")
    dark_mag  = vv.subtract(mean_vv).multiply(-1).divide(std_vv).rename("dark_spot_magnitude")
    vv_lin    = ee.Image(10).pow(vv.divide(10))
    vh_lin    = ee.Image(10).pow(vh.divide(10))
    ratio     = vv_lin.divide(vh_lin).rename("vv_vh_ratio")
    return image.addBands(dark_mask).addBands(dark_mag).addBands(ratio)

def build_s1_composite(roi, start, end, sigma=1.5):
    col   = get_s1_collection(roi, start, end)
    count = col.size().getInfo()
    print(f"  S1 scenes ({start} to {end}): {count}")
    if count == 0:
        return None, None, 0
    col = col.map(lambda img: compute_sar_features(img, roi, sigma))
    return col.median().clip(roi), col, count

def compute_sar_change(s1_base, s1_rec, roi, threshold_db=-3.0):
    change        = s1_rec.select("VV").subtract(s1_base.select("VV")).rename("SAR_change_dB")
    new_spill     = change.lt(threshold_db).rename("new_spill_mask")
    chronic_spill = s1_base.select("dark_spot_mask").And(s1_rec.select("dark_spot_mask")).rename("chronic_spill_mask")
    return change.addBands(new_spill).addBands(chronic_spill)

print("All functions defined.")


All functions defined.


## Cell 5 — Run Sentinel-2 vegetation analysis

In [5]:
print("=== Sentinel-2 Vegetation Analysis ===")

print("Baseline composite...")
s2_baseline, n_s2_base = build_s2_composite(ROI, BASELINE_START, BASELINE_END)

print("Recent composite...")
s2_recent, n_s2_rec = build_s2_composite(ROI, RECENT_START, RECENT_END)

if s2_baseline and s2_recent:
    ndvi_change = s2_recent.select("NDVI").subtract(s2_baseline.select("NDVI")).rename("NDVI_change")
    veg_loss    = ndvi_change.lt(-0.15).rename("vegetation_loss_mask")
    ndvi_anomaly = ndvi_change.addBands(veg_loss)

    base_stats = s2_baseline.select("NDVI").reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True
    ).getInfo()
    rec_stats = s2_recent.select("NDVI").reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True
    ).getInfo()

    base_mean = base_stats.get("NDVI_mean", 0) or 0
    rec_mean  = rec_stats.get("NDVI_mean",  0) or 0
    delta     = rec_mean - base_mean
    direction = "DECREASE (vegetation loss)" if delta < 0 else "INCREASE (recovery)"

    print()
    print("NDVI statistics:")
    print(f"  Baseline mean : {base_mean:.3f}")
    print(f"  Recent mean   : {rec_mean:.3f}")
    print(f"  Change        : {delta:+.3f}  ->  {direction}")
else:
    ndvi_anomaly = None
    print("No S2 composites — skipping vegetation analysis")


=== Sentinel-2 Vegetation Analysis ===
Baseline composite...
  S2 scenes (2023-01-01 to 2023-06-30): 56
Recent composite...
  S2 scenes (2024-01-01 to 2024-06-30): 48

NDVI statistics:
  Baseline mean : 0.462
  Recent mean   : 0.509
  Change        : +0.047  ->  INCREASE (recovery)


## Cell 6 — Run SAR change detection

In [6]:
print("=== SAR Change Detection ===")

s1_baseline, s1_baseline_col, n_s1_base = build_s1_composite(ROI, BASELINE_START, BASELINE_END)
s1_recent,   s1_recent_col,   n_s1_rec  = build_s1_composite(ROI, RECENT_START,   RECENT_END)

sar_change = compute_sar_change(s1_baseline, s1_recent, ROI)

change_stats = sar_change.select("SAR_change_dB").reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True)
                            .combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True
).getInfo()

new_spill_px = sar_change.select("new_spill_mask").reduceRegion(
    reducer=ee.Reducer.sum(), geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True
).getInfo()
chronic_px = sar_change.select("chronic_spill_mask").reduceRegion(
    reducer=ee.Reducer.sum(), geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True
).getInfo()

print()
print("SAR change (dB):")
print(f"  Mean   : {change_stats.get(chr(83)+chr(65)+chr(82)+chr(95)+chr(99)+chr(104)+chr(97)+chr(110)+chr(103)+chr(101)+chr(95)+chr(100)+chr(66)+chr(95)+chr(109)+chr(101)+chr(97)+chr(110), 0):.2f} dB")
print(f"  StdDev : {change_stats.get(chr(83)+chr(65)+chr(82)+chr(95)+chr(99)+chr(104)+chr(97)+chr(110)+chr(103)+chr(101)+chr(95)+chr(100)+chr(66)+chr(95)+chr(115)+chr(116)+chr(100)+chr(68)+chr(101)+chr(118), 0):.2f} dB")
print(f"  New spill pixels  : {int(new_spill_px.get(chr(110)+chr(101)+chr(119)+chr(95)+chr(115)+chr(112)+chr(105)+chr(108)+chr(108)+chr(95)+chr(109)+chr(97)+chr(115)+chr(107), 0) or 0):,}")
print(f"  Chronic spill px  : {int(chronic_px.get(chr(99)+chr(104)+chr(114)+chr(111)+chr(110)+chr(105)+chr(99)+chr(95)+chr(115)+chr(112)+chr(105)+chr(108)+chr(108)+chr(95)+chr(109)+chr(97)+chr(115)+chr(107), 0) or 0):,}")


=== SAR Change Detection ===
  S1 scenes (2023-01-01 to 2023-06-30): 30
  S1 scenes (2024-01-01 to 2024-06-30): 29

SAR change (dB):
  Mean   : -0.01 dB
  StdDev : 0.71 dB
  New spill pixels  : 1,377
  Chronic spill px  : 11,685


## Cell 7 — DBSCAN fire hotspot clustering

Groups the 50 VIIRS hotspots into spatially distinct refinery site candidates.
 degrees ~ 5.5 km at Niger Delta latitude.
Points labelled -1 are noise (isolated fires, likely agricultural).

In [7]:
print('=== DBSCAN Clustering ===')
print()

if df_firms.empty:
    print('No FIRMS data.')
    df_clusters = pd.DataFrame()
else:
    coords  = df_firms[['lat','lon']].values
    eps_rad = np.radians(0.05)
    db      = DBSCAN(eps=eps_rad, min_samples=2,
                     algorithm='ball_tree', metric='haversine').fit(np.radians(coords))
    df_firms['cluster'] = db.labels_

    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise    = (db.labels_ == -1).sum()
    print('Hotspots     : ' + str(len(df_firms)))
    print('Clusters     : ' + str(n_clusters))
    print('Noise points : ' + str(n_noise) + ' (isolated fires)')
    print()

    summary = []
    for cid in sorted(set(db.labels_)):
        if cid == -1:
            continue
        g    = df_firms[df_firms['cluster'] == cid]
        src  = g['likely_source'].value_counts().index[0]
        risk = 'HIGH' if src == 'refinery_candidate' else 'MEDIUM' if src == 'flare_candidate' else 'LOW'
        r = {
            'cluster_id':       cid,
            'n_hotspots':       len(g),
            'centroid_lat':     g['lat'].mean(),
            'centroid_lon':     g['lon'].mean(),
            'mean_T21_K':       g['T21_max_K'].mean(),
            'max_T21_K':        g['T21_max_K'].max(),
            'mean_fire_count':  g['fire_count'].mean(),
            'dominant_source':  src,
            'risk_label':       risk,
        }
        summary.append(r)
        print('  Cluster ' + str(cid) + ': ' + str(len(g)) + ' hotspots'
              + '  lat=' + str(round(r['centroid_lat'], 3))
              + '  lon=' + str(round(r['centroid_lon'], 3))
              + '  T21=' + str(round(r['max_T21_K'], 0)) + 'K'
              + '  risk=' + risk)

    df_clusters = pd.DataFrame(summary)
    print()
    print('Cluster table: ' + str(len(df_clusters)) + ' sites')

=== DBSCAN Clustering ===

Hotspots     : 50
Clusters     : 8
Noise points : 7 (isolated fires)

  Cluster 0: 2 hotspots  lat=5.089  lon=6.513  T21=349.0K  risk=LOW
  Cluster 1: 12 hotspots  lat=5.715  lon=6.528  T21=349.0K  risk=LOW
  Cluster 2: 2 hotspots  lat=5.282  lon=6.813  T21=341.0K  risk=LOW
  Cluster 3: 3 hotspots  lat=5.561  lon=6.876  T21=332.0K  risk=LOW
  Cluster 4: 2 hotspots  lat=5.038  lon=6.948  T21=342.0K  risk=LOW
  Cluster 5: 4 hotspots  lat=5.307  lon=7.007  T21=346.0K  risk=LOW
  Cluster 6: 3 hotspots  lat=5.24  lon=7.069  T21=338.0K  risk=LOW
  Cluster 7: 15 hotspots  lat=5.187  lon=7.155  T21=347.0K  risk=LOW

Cluster table: 8 sites


## Cell 8 — Cluster scatter plot

In [8]:
# Cluster plot skipped — matplotlib crashes kernel on this machine
# The cluster data is saved to CSV below — visualised in Module 4 dashboard instead

if len(df_clusters) > 0:
    print('Cluster summary:')
    print(df_clusters[['cluster_id','n_hotspots','centroid_lat',
                        'centroid_lon','max_T21_K','risk_label']].to_string(index=False))
else:
    print('No clusters.')

Cluster summary:
 cluster_id  n_hotspots  centroid_lat  centroid_lon  max_T21_K risk_label
          0           2      5.089237      6.513347 349.299988        LOW
          1          12      5.715240      6.528333 348.899994        LOW
          2           2      5.282094      6.813160 341.000000        LOW
          3           3      5.560586      6.876416 332.500000        LOW
          4           2      5.037864      6.947907 341.799988        LOW
          5           4      5.307443      7.007196 346.299988        LOW
          6           3      5.239985      7.068618 338.299988        LOW
          7          15      5.187098      7.154738 346.899994        LOW


## Cell 9 — Interactive map with all Module 2 layers

In [9]:
import geemap

m = geemap.Map(center=ROI_CENTER, zoom=ROI_ZOOM)
m.add_basemap("SATELLITE")

m.addLayer(sar_change.select("SAR_change_dB"),
    {"min": -10, "max": 10,
     "palette": ["E24B4A","FAC775","ffffff","9FE1CB","085041"]},
    "SAR change dB (red=new dark, green=new bright)")

m.addLayer(sar_change.select("new_spill_mask").selfMask(),
    {"palette": ["E24B4A"], "opacity": 0.8},
    "New spill candidates (< -3dB)")

m.addLayer(sar_change.select("chronic_spill_mask").selfMask(),
    {"palette": ["854F0B"], "opacity": 0.7},
    "Chronic spill zones")

if ndvi_anomaly:
    m.addLayer(s2_recent.select("NDVI"),
        {"min": -0.2, "max": 0.8,
         "palette": ["E24B4A","FAC775","ffffff","C0DD97","27500A"]},
        "NDVI recent (red=bare, green=healthy)")
    m.addLayer(ndvi_anomaly.select("vegetation_loss_mask").selfMask(),
        {"palette": ["D85A30"], "opacity": 0.75},
        "Vegetation loss (NDVI drop > 0.15)")

if not df_firms.empty and "cluster" in df_firms.columns:
    pts = [ee.Feature(ee.Geometry.Point([r["lon"], r["lat"]]),
                      {"cluster": int(r["cluster"]),
                       "T21_max_K": float(r["T21_max_K"]),
                       "source": str(r["likely_source"])})
           for _, r in df_firms.iterrows()]
    m.addLayer(ee.FeatureCollection(pts), {"color": "EF9F27"}, "FIRMS clusters")

m.addLayer(ee.FeatureCollection([ee.Feature(ROI)]),
    {"color": "185FA5", "fillColor": "00000000"}, "TNP boundary")

m.add_layer_control()
print("Map ready.")
m


Map ready.


Map(center=[5.4, 6.85], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

## Cell 10 — Build and export feature table for Module 3

In [11]:
print('=== Building Feature Table ===')

# Stack all available bands
feature_stack = s1_recent.select(['VV','VH','dark_spot_mask','dark_spot_magnitude'])

if ndvi_anomaly:
    feature_stack = feature_stack.addBands(s2_recent.select(['NDVI','NDWI','NDMI']))
    feature_stack = feature_stack.addBands(ndvi_anomaly.select('NDVI_change'))

feature_stack = feature_stack.addBands(
    sar_change.select(['SAR_change_dB','new_spill_mask','chronic_spill_mask'])
)

print('Sampling feature stack at 5km grid (300 points)...')
sampled = feature_stack.sample(
    region=ROI, scale=5000, numPixels=300,
    geometries=True, seed=42
)

sample_info = sampled.getInfo()
rows = []
for feat in sample_info['features']:
    coords = feat['geometry']['coordinates']
    props  = feat['properties']
    props['lon'] = coords[0]
    props['lat'] = coords[1]
    rows.append(props)

df_features = pd.DataFrame(rows)
print('Feature table: ' + str(len(df_features)) + ' rows x ' + str(len(df_features.columns)) + ' columns')
print('Columns: ' + str(df_features.columns.tolist()))
print()

# Save all outputs
feat_csv = os.path.join(CACHE_DIR, 'm2_feature_table.csv')
df_features.to_csv(feat_csv, index=False)
print('Feature table     -> ' + feat_csv + '  (' + str(round(os.path.getsize(feat_csv)/1024, 1)) + ' KB)')

df_firms.to_csv(os.path.join(CACHE_DIR, 'm2_hotspots_clustered.csv'), index=False)
print('Clustered hotspots -> data/cached/m2_hotspots_clustered.csv')

if len(df_clusters) > 0:
    df_clusters.to_csv(os.path.join(CACHE_DIR, 'm2_refinery_clusters.csv'), index=False)
    print('Cluster summary    -> data/cached/m2_refinery_clusters.csv')

meta = {
    'module':       'M2_processing',
    'exported':     datetime.now().isoformat(),
    'feature_rows': len(df_features),
    'feature_cols': len(df_features.columns),
    'fire_clusters': len(df_clusters) if len(df_clusters) > 0 else 0,
}
with open(os.path.join(CACHE_DIR, 'm2_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('Metadata           -> data/cached/m2_metadata.json')
print()
print('All Module 2 outputs written. Commit data/cached/ to GitHub.')

=== Building Feature Table ===
Sampling feature stack at 5km grid (300 points)...
Feature table: 270 rows x 13 columns
Columns: ['NDMI', 'NDVI', 'NDVI_change', 'NDWI', 'SAR_change_dB', 'VH', 'VV', 'chronic_spill_mask', 'dark_spot_magnitude', 'dark_spot_mask', 'new_spill_mask', 'lon', 'lat']

Feature table     -> ../data/cached\m2_feature_table.csv  (53.4 KB)
Clustered hotspots -> data/cached/m2_hotspots_clustered.csv
Cluster summary    -> data/cached/m2_refinery_clusters.csv
Metadata           -> data/cached/m2_metadata.json

All Module 2 outputs written. Commit data/cached/ to GitHub.


## Module 2 complete

| Output | File | Used in |
|--------|------|---------|
| Multi-signal feature table |  | Module 3 ML |
| Refinery site clusters |  | Module 4 dashboard |
| Clustered hotspots |  | Module 4 dashboard |
| DBSCAN cluster map |  | README / portfolio |

**Next: Module 3 — ML Anomaly Detection**
- XGBoost fire + SO2 risk scorer trained on the feature table above
- Isolation Forest anomaly detector for unsupervised outlier flagging
- SHAP feature importance — shows which sensor drives each alert
- Output: risk-scored feature table for Module 4 dashboard